In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score

In [ ]:
# 1. بارگذاری داده‌ها
train_df = pd.read_csv(r'train.csv')
test_df = pd.read_csv(r'test.csv')

print("داده‌های آموزشی بارگذاری شدند.")
print(f"شکل (Shape) مجموعه داده آموزش: {train_df.shape}")
print(f"شکل (Shape) مجموعه داده تست: {test_df.shape}\n")

In [ ]:
# 2. جدا کردن ویژگی‌ها و هدف
X = train_df.drop(columns=['electricity_consumption_kWh'])
y = train_df['electricity_consumption_kWh']

In [ ]:
# 3. ایجاد یک خط لوله (Pipeline) برای پیش‌پردازش و مدل‌سازی
# این خط لوله ابتدا مقادیر خالی را با میانگین پر می‌کند و سپس داده‌ها را مقیاس‌بندی می‌کند
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])


In [ ]:
# 4. تقسیم داده‌ها برای ارزیابی (Validation)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# 5. آموزش مدل با استفاده از Pipeline
pipeline.fit(X_train, y_train)

In [ ]:
# 6. ارزیابی مدل
y_val_pred = pipeline.predict(X_val)
r2_val = r2_score(y_val, y_val_pred)
print(f"R² Score روی مجموعه اعتبارسنجی (با پاکسازی و نرمال‌سازی): {r2_val:.4f}")

In [ ]:
# 7. پیش‌بینی نهایی روی داده‌های تست
# تست‌ست به صورت خودکار توسط pipeline (imputer & scaler) پردازش می‌شود
y_test_pred = pipeline.predict(test_df)

In [ ]:
# 8. ذخیره فایل خروجی
submission_df = pd.DataFrame({'electricity_consumption_kWh': y_test_pred})
submission_df.to_csv('submission_linear_regression_optimized.csv', index=False)

print("\nفایل submission_linear_regression_optimized.csv با موفقیت ساخته شد.")

In [ ]:
import joblib

# ۱. استخراج و ذخیره اسکیلر استاندارد از لایه دوم پایپ‌لاین
extracted_scaler = pipeline.named_steps['scaler']
joblib.dump(extracted_scaler, 'scaler.joblib')
print("✅ اسکیلر (StandardScaler) با موفقیت از پایپ‌لاین استخراج و ذخیره شد!")

# ۲. استخراج و ذخیره مدل رگرسیون خطی از لایه سوم پایپ‌لاین
extracted_model = pipeline.named_steps['regressor']
joblib.dump(extracted_model, 'linear_model.joblib')
print("✅ مدل رگرسیون خطی با موفقیت از پایپ‌لاین استخراج و ذخیره شد!")